# 0 — Data and Baseline

The two supplied WI LGPIF CSV files retain the published 4,991/1,039 claim split.
Sources: [Gan and Shultz (2026)](https://doi.org/10.66573/001c.162423) and
[original code](https://github.com/ganml/llmlgpif).

Environment: Python 3.12.0; numpy 2.4.3; pandas 2.3.3; scipy 1.17.1; statsmodels 0.14.6; scikit-learn 1.8.0; matplotlib 3.10.8; seaborn 0.13.2; nbformat 5.10.4.

Run notebooks 0–6 in order from this directory. This notebook creates `intermediate/`;
later notebooks read earlier results there. Tables are saved as pandas pickle files
to preserve their types, row order, and numerical precision.


In [1]:
from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd
import statsmodels.api as sm

DATA = Path.cwd()
INTERMEDIATE = DATA / "intermediate"
INTERMEDIATE.mkdir(exist_ok=True)

PERILS = [
    "Vandalism",
    "Fire",
    "Lightning",
    "Wind",
    "Hail",
    "Vehicle",
    "WaterNW",
    "WaterW",
    "Misc",
]
FEATURES = PERILS[:-1]


def prepare(frame, split):
    frame = frame.copy()
    frame["Description"] = frame.Description.fillna("").astype(str).str.strip()
    frame["split"] = split
    frame["claim_type"] = frame[PERILS].idxmax(axis=1)
    frame["log_loss"] = np.log1p(frame.Loss)
    return frame


train = prepare(pd.read_csv(DATA / "peril.training.csv"), "train")
valid = prepare(pd.read_csv(DATA / "peril.validation.csv"), "validation")
claims = pd.concat([train, valid], ignore_index=True)
claims.insert(0, "claim_row", np.arange(len(claims)))
descriptions = claims[["Description"]].drop_duplicates().reset_index(drop=True)

## Sample and Peril Counts

In [2]:
sample_summary = pd.DataFrame(
    [
        {
            "split": split,
            "claims": len(frame),
            "unique_descriptions": frame.Description.nunique(),
            "median_loss": frame.Loss.median(),
            "mean_loss": frame.Loss.mean(),
        }
        for split, frame in [
            ("train", train),
            ("validation", valid),
            ("combined", claims),
        ]
    ]
)
display(sample_summary.round(2))

peril_summary = pd.DataFrame(index=PERILS)
for split, frame in [("train", train), ("validation", valid), ("combined", claims)]:
    peril_summary[split + "_count"] = frame[PERILS].sum()
    peril_summary[split + "_percent"] = 100 * frame[PERILS].mean()
display(peril_summary.round(2))

,split,claims,unique_descriptions,median_loss,mean_loss
0,train,4991,3733,2243.28,19297.05
1,validation,1039,996,3240.00,21488.37
2,combined,6030,4690,2392.01,19674.63


,train_count,train_percent,validation_count,validation_percent,combined_count,combined_percent
Vandalism,1774,35.54,310,29.84,2084,34.56
Fire,171,3.43,46,4.43,217,3.60
Lightning,832,16.67,123,11.84,955,15.84
Wind,296,5.93,107,10.30,403,6.68
Hail,76,1.52,18,1.73,94,1.56
Vehicle,852,17.07,227,21.85,1079,17.89
WaterNW,202,4.05,67,6.45,269,4.46
WaterW,426,8.54,38,3.66,464,7.69
Misc,362,7.25,103,9.91,465,7.71


## Replicated no-LLM Baseline

Fit a Gamma GLM with log link to `log1p(Loss)`, using eight peril indicators
and Miscellaneous as the reference. Validation MAE and R² are on the transformed
loss scale, as in the original study.

In [3]:
def regression_metrics(y, prediction, p):
    y, prediction = np.asarray(y), np.asarray(prediction)
    r2 = 1 - np.square(y - prediction).sum() / np.square(y - y.mean()).sum()
    return {
        "r2": r2,
        "adjusted_r2": 1 - (1 - r2) * (len(y) - 1) / (len(y) - p - 1),
        "mae": np.abs(y - prediction).mean(),
    }


peril_design = sm.add_constant(claims[FEATURES], has_constant="add")
train_mask = claims.split.eq("train")
baseline_fit = sm.GLM(
    claims.loc[train_mask, "log_loss"],
    peril_design.loc[train_mask],
    family=sm.families.Gamma(link=sm.families.links.Log()),
).fit()

baseline_prediction = pd.Series(index=claims.index, dtype=float)
baseline_rows = []
for split in ["train", "validation"]:
    mask = claims.split.eq(split)
    prediction = baseline_fit.predict(peril_design.loc[mask])
    baseline_prediction.loc[mask] = prediction
    baseline_rows.append(
        {
            "split": split,
            **regression_metrics(
                claims.loc[mask, "log_loss"], prediction, len(FEATURES)
            ),
        }
    )
baseline = pd.DataFrame(baseline_rows)
baseline_predictions = claims[["claim_row"]].copy()
baseline_predictions["prediction"] = baseline_prediction.to_numpy()
baseline_predictions["abs_error"] = np.abs(claims.log_loss - baseline_prediction)
display(baseline.round(4))

,split,r2,adjusted_r2,mae
0,train,0.3742,0.3732,1.0585
1,validation,0.3398,0.3347,1.0694


## Save results

In [4]:
pd.to_pickle(
    {
        "claims": claims,
        "descriptions": descriptions,
        "perils": PERILS,
        "features": FEATURES,
        "peril_design": peril_design,
        "baseline": baseline,
        "baseline_predictions": baseline_predictions,
        "sample_summary": sample_summary,
        "peril_summary": peril_summary,
    },
    INTERMEDIATE / "s0_data.pkl",
)